# 07 Camera model: video -> body keypoints -> activity
Our own recordings are phone **videos** of 4 people (p1-p4), with no sensor data. MediaPipe Pose turns each video into 33 body landmarks per frame (`src/video_pose.py`); a small 1D-CNN classifies 2 s windows.

Classes: walking, standing, stairs (up and down merged: the direction cannot be told from the side views). There are no sitting or lying clips, so these numbers are **not comparable** with the 6-class phone model.

With only 4 people, evaluation is **leave-one-person-out**: train on 3 people, test on the 4th, four times. In each fold one of the 3 training people is the validation set for early stopping.

In [1]:
import sys, yaml, numpy as np, pandas as pd
sys.path.insert(0, "../src")
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from video_pose import load_video
from models import build_cnn, build_rf
from train import fit
cfg = yaml.safe_load(open("../config/config.yaml"))
CL = cfg["video"]["classes"]; FIG = "../reports/figures"
d = load_video("../data/video/keypoints", cfg)
X, y, sub = d["X"], d["y"], d["subject"]
print(X.shape)
pd.crosstab(pd.Series(sub, name="person"), pd.Series([CL[i] for i in y], name="class"))

(493, 50, 66)


class,stairs,standing,walking
person,,,
p1,54,68,20
p2,15,20,64
p3,31,70,18
p4,32,34,47
p5,0,0,20


In [2]:
# p5 has walking clips only: used for training, never as test or validation person
def mirror(X):  # camera side is arbitrary: flip x (even channels) as augmentation
    Z = X.copy(); Z[..., 0::2] *= -1; return np.concatenate([X, Z])

def feats(X):   # for the Random Forest baseline
    return np.concatenate([X.mean(1), X.std(1), X.min(1), X.max(1)], 1)

# Two inputs: x,y of every landmark, or height (y) only. x changes completely between front/side/back
# views; y barely does, so y-only should generalise better across people filmed from different angles.
VARIANTS = {"x,y": lambda A: A, "y only": lambda A: A[..., 1::2]}
people = sorted(set(sub) - {"p5"}); rows, preds = [], {}
for vn, sel in VARIANTS.items():
    for t in people:
        va = people[(people.index(t) + 1) % len(people)]
        te, vm = sub == t, sub == va; tr = ~te & ~vm
        aug = mirror if vn == "x,y" else (lambda A: A)
        Xtr = aug(sel(X[tr])); ytr = np.concatenate([y[tr]] * (2 if vn == "x,y" else 1))
        m, _ = fit(build_cnn((X.shape[1], sel(X[:1]).shape[2]), len(CL)), Xtr, ytr, sel(X[vm]), y[vm], cfg)
        p = m.predict(sel(X[te]), verbose=0).argmax(1)
        rf, _ = fit(build_rf(), feats(sel(X[~te])), y[~te], None, None, cfg); q = rf.predict(feats(sel(X[te])))
        for mn, z in [("CNN", p), ("RF", q)]:
            rows.append({"input": vn, "model": mn, "test person": t, "acc": accuracy_score(y[te], z),
                         "macro F1": f1_score(y[te], z, average="macro")})
            preds.setdefault((vn, mn), []).append((y[te], z))
res = pd.DataFrame(rows)
summ = res.groupby(["input", "model"])[["acc", "macro F1"]].agg(["mean", "std"]).round(3)
summ

Epoch 1/30


8/8 - 1s - 175ms/step - accuracy: 0.4067 - loss: 1.1836 - val_accuracy: 0.2020 - val_loss: 1.3077


Epoch 2/30


8/8 - 0s - 20ms/step - accuracy: 0.4940 - loss: 0.9513 - val_accuracy: 0.2525 - val_loss: 1.4251


Epoch 3/30


8/8 - 0s - 18ms/step - accuracy: 0.5496 - loss: 0.8619 - val_accuracy: 0.2828 - val_loss: 1.7412


... (181 outputs left out)

4/4 - 0s - 30ms/step - accuracy: 0.7521 - loss: 0.5835 - val_accuracy: 0.7324 - val_loss: 0.6051


Epoch 30/30


4/4 - 0s - 28ms/step - accuracy: 0.7647 - loss: 0.5364 - val_accuracy: 0.7606 - val_loss: 0.6185


acc        macro F1       
               mean    std     mean    std
input  model                              
x,y    CNN    0.456  0.150    0.407  0.071
       RF     0.586  0.159    0.589  0.151
y only CNN    0.373  0.094    0.340  0.104
       RF     0.459  0.051    0.472  0.085

In [3]:
fig, axs = plt.subplots(2, 2, figsize=(10, 8))
for ax, ((vn, mn), L) in zip(axs.ravel(), preds.items()):
    yt = np.concatenate([a for a, _ in L]); yp = np.concatenate([b for _, b in L])
    ConfusionMatrixDisplay(confusion_matrix(yt, yp, labels=range(len(CL))), display_labels=CL).plot(ax=ax, colorbar=False)
    ax.set_title(f"{mn}, input {vn} (all 4 folds)")
fig.tight_layout(); fig.savefig(f"{FIG}/video_confusion.png", dpi=120)
res.pivot_table(index="test person", columns=["input", "model"], values="macro F1").round(3)

input          x,y        y only       
model          CNN     RF    CNN     RF
test person                            
p1           0.434  0.785  0.347  0.525
p2           0.426  0.567  0.223  0.527
p3           0.304  0.589  0.474  0.489
p4           0.465  0.417  0.315  0.346

In [4]:
# final CNN on all people, early-stopped on a random 15% of windows. NOT used by the app (it uses the RF from notebook 08).
# Windows overlap 50%, so this validation set shares frames with training: its validation accuracy is inflated, don't quote it.
# BEST = input variant with the highest mean macro F1 for the CNN
BEST = summ.xs("CNN", level="model")[("macro F1", "mean")].idxmax(); sel = VARIANTS[BEST]; print("final CNN input:", BEST)
rng = np.random.default_rng(cfg["seed"]); v = rng.random(len(X)) < 0.15
aug = mirror if BEST == "x,y" else (lambda A: A)
final, _ = fit(build_cnn((X.shape[1], sel(X[:1]).shape[2]), len(CL)), aug(sel(X[~v])),
               np.concatenate([y[~v]] * (2 if BEST == "x,y" else 1)), sel(X[v]), y[v], cfg)
final.save("../models/video_cnn.keras"); print("saved")

final CNN input: x,y
Epoch 1/30


13/13 - 1s - 99ms/step - accuracy: 0.4224 - loss: 1.0711 - val_accuracy: 0.5057 - val_loss: 0.9922


Epoch 2/30


13/13 - 0s - 13ms/step - accuracy: 0.5591 - loss: 0.9124 - val_accuracy: 0.5287 - val_loss: 0.9077


Epoch 3/30


13/13 - 0s - 14ms/step - accuracy: 0.5813 - loss: 0.8296 - val_accuracy: 0.5402 - val_loss: 0.8678


... (25 outputs left out)

13/13 - 0s - 14ms/step - accuracy: 0.9236 - loss: 0.2064 - val_accuracy: 0.7931 - val_loss: 0.4481


Epoch 17/30


13/13 - 0s - 14ms/step - accuracy: 0.9113 - loss: 0.2077 - val_accuracy: 0.7471 - val_loss: 0.6436


saved
